# CS 467 Humidity Sensor

Filtering and LED pattern conversion prototyping

Algorithm prototypes are implemented in Python, but production algorithms will be implemented in Rust

## Requirements

- Install Python 3.13
- Install `uv`: https://docs.astral.sh/uv/

## Usage

- Install dependencies with `uv` by running `uv sync`
  - This will create a virtual environment in the `.venv` directory with all of the necessary dependencies installed
- Open the notebook file and select the interpreter in `.venv`

## Imports

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import plotly.express as px
import plotly.graph_objects as go
import random
import ipytest
ipytest.autoconfig()

## Generate Sample Data

In [ ]:
random.seed(42)

data = []
for i in range(1, 100):
    data.extend([i] * random.randint(10, 200))

df = pd.DataFrame(data, columns=["Clean Data"])
df.index.name = "Time"

### Add Noise

In [ ]:
mu = 0
sigma = 0.5
noise = np.random.normal(mu, sigma, size=len(data)).round(1)

df["Noisy Data"] = df["Clean Data"] + noise

In [ ]:
display(df)

In [ ]:
LAYOUT_PARAMS = {
    "height": 600,
    "width": 800,
    "font": dict(size=18),
    "xaxis_title": "Time (s)",
    "yaxis_title": "Humidity (%)",
    "legend_title_text": None
}

In [ ]:
fig = px.line(
    df, x=df.index, y=["Noisy Data", "Clean Data"], title="Simulated Readings"
)
fig.update_traces(line=dict(width=3), selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), selector={"name": "Noisy Data"})

for threshold in [20, 40, 50, 60, 70]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1
    )

fig.update_layout(dict1=LAYOUT_PARAMS)

display(fig)

In [ ]:
x_range = range(1600, 2200)

fig = px.line(
    df.iloc[x_range], x=df.index[x_range], y=["Noisy Data", "Clean Data"], title="Simulated Readings (Zoomed)"
)
fig.update_traces(line=dict(width=3), selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), selector={"name": "Noisy Data"})

fig.update_layout(LAYOUT_PARAMS)

for threshold in [20]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1,
        # annotation_text="",
        # annotation_position="bottom right"
    )

display(fig)

## Implement Filtering Methods

### Rolling Average

- **Pros**
  - Simple and straightforward to implement
  - Changing the window size is intuitive
- **Cons**
  - Weights newer samples the same as older ones
  - Requires holding the previous $n$ samples in memory
    - Changing the window size also changes the number of samples required in memory

In [ ]:
df["Filtered Data (Rolling)"] = df["Noisy Data"].rolling(60).mean().round(1)

### Exponential Weighted Mean

$$
\begin{align}
y_0 &= x_0 \\\\
y_t &= (1 - \alpha) y_{t-1} + \alpha x_t 
\end{align}
$$

- **Pros**
  - Weights newer samples more than older samples
  - Only requires holding the current sample and the previous filter output in memory
    - Tuning the weighting parameter does not change the memory requirements
- **Cons**
  - Less intuitive than a rolling average in the sense that the contribution of each sample to the filter output is not immediatley obvious

- Also to note, this is used in TCP implementations to measure network latency
  - The measured packet round trip time (RTT) is EWM-filtered (with a different $\alpha$)

In [ ]:
df["Filtered Data (EWM)"] = df["Noisy Data"].ewm(alpha=0.015, adjust=False).mean().round(1)

In [ ]:
x_range = range(4600, 5100)

fig = px.line(
    df.iloc[x_range],
    x=df.index[x_range],
    y=["Noisy Data", "Clean Data", "Filtered Data (Rolling)", "Filtered Data (EWM)"],
    title="Simulated Readings (Zoomed)",
)
fig.update_traces(line=dict(width=2), opacity=0.8, selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), opacity=0.8, selector={"name": "Noisy Data"})
fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (Rolling)"},
)
fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (EWM)"},
)

for threshold in [50]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1
    )

fig.update_layout(LAYOUT_PARAMS)

display(fig)

## Implement Measurement to LED Pattern Converter

In [ ]:
def render_pattern(
    value: float, thresholds: list[int] = [0, 20, 40, 50, 60, 70]
) -> list[bool]:
    """
    Render a value as a discrete pattern.

    Arguments:
        value: the value to render
        thresholds: the pattern cutoff values

    Returns
        The pattern as a list of boolean states.
    """
    pattern = [None] * len(thresholds)

    for i, threshold in enumerate(thresholds):
        pattern[i] = True if value >= threshold else False

    return pattern

### Test Rendering 

In [ ]:
%%ipytest

def test_negative():
    assert render_pattern(-1) == [False] * 6

def test_large_value():
    assert render_pattern(100000) == [True] * 6

def test_threshold_low_boundary():
    assert render_pattern(40) == [True, True, True, False, False, False]

def test_threshold_middle():
    assert render_pattern(45) == [True, True, True, False, False, False]

def test_threshold_high_boundary():
    assert render_pattern(49) == [True, True, True, False, False, False]


### Example

## Convert the Noisy and Filtered Sample Data

- To visualize the results the patterns will be represented as integers
  - [True, True, False, False, False, False] --> 2

In [ ]:
# This is just a helper for converting the values
# Not intended as a production prototype

def render_pattern_sum_helper(value: float) -> int:
    return sum(render_pattern(value))

In [ ]:
df["Noisy Pattern"] = df["Noisy Data"].apply(render_pattern_sum_helper)
df["Filtered Pattern (Rolling)"] = df["Filtered Data (Rolling)"].apply(render_pattern_sum_helper)
df["Filtered Pattern (EWM)"] = df["Filtered Data (EWM)"].apply(render_pattern_sum_helper)

## Visualize the Patterns

In [ ]:
fig = px.line(
    df,
    x=df.index,
    y=[
        "Noisy Pattern",
        "Filtered Pattern (Rolling)",
        "Filtered Pattern (EWM)",
    ],
    title="Simulated Patterns",
)
fig.update_traces(line=dict(width=2), selector={"name": "Clean Data"})
fig.update_traces(line=dict(width=2), selector={"name": "Noisy Data"})

fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (Rolling)"},
)
fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Data (EWM)"},
)

fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_yaxes(title="Pattern Number")

display(fig)

In [ ]:
fig = px.line(
    df.loc[x_range],
    x=df.index[x_range],
    y=[
        "Noisy Pattern",
        "Filtered Pattern (Rolling)",
        "Filtered Pattern (EWM)",
    ],
    title="Simulated Patterns (Zoomed)",
)

fig.update_traces(line=dict(width=1), opacity=0.5, selector={"name": "Noisy Pattern"})
fig.update_traces(line=dict(width=2), selector={"name": "Filtered Pattern (Rolling)"})
fig.update_traces(line=dict(width=2), selector={"name": "Filtered Pattern (EWM)"})

fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_yaxes(title="Pattern Number", dtick=1)

display(fig)

## Conclusion

- The sensor data should be filtered to avoid rapid oscillations near the pattern boundaries
- The actual filter noise will need to be characterized and then used to tune the production filter
- EWM and rolling average methods are both viable (after tuning), but EWM is more space efficient

## Characterize Real Sensor Noise

- `raw_humidity.csv` contains 14 hours of humidity readings sampled at 0.5 Hz
- To characterize the noise, the data is read and filtered with a rolling average
- The filtered signal is then subtracted from the raw signal
- The residual is the approximate noise of the sensor
- The noise statistics are then characterized over an approximately steady state period
- The EWM (IIR) filter parameter is then chosen based on the noise statistics and desired responsiveness 

In [ ]:
df = pd.read_csv("Data Processing Prototyping/raw_humidity.csv", skipinitialspace=True)

# Create a date time index relative to the unix epoch
# Plotly can use this to nicely format the x axis ticks
time_delta = pd.to_timedelta(df["Elapsed Time (ms)"], unit='ms')
BASE_DATE = pd.Timestamp("1970/01/01")
df["Date Time"] = BASE_DATE + time_delta
df = df.set_index("Date Time")


# Filter the signal and subtract the result from the measurements to approximately
# isolate the residual noise for statistical analysis
df["Smoothed Humidity (%)"] = df["Humidity (%)"].rolling(window=30).mean()
df["Noise (%)"] = df["Humidity (%)"] - df["Smoothed Humidity (%)"]

display(df)

In [ ]:
fig = px.line(
    df, x=df.index, y=["Humidity (%)"], title="Humidity Readings"
)

for threshold in [20, 40, 50, 60, 70]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1
    )
fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_layout(xaxis_title="Elapsed Time", xaxis_tickformat="%H:%M:%S")

display(fig)

In [ ]:
steady_state_df = df.between_time('04:00', '12:00')
fig = px.line(
    steady_state_df, x=steady_state_df.index, y=["Noise (%)"], title="Approximate Steady State Noise"
)
fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_layout(xaxis_title="Elapsed Time", xaxis_tickformat="%H:%M:%S")

display(fig)
steady_state_df["Noise (%)"].describe()

In [ ]:
fig = px.histogram(
    steady_state_df["Noise (%)"], histnorm="probability density", title="Humidity Error Distribution"
)
fig.update_layout(
    LAYOUT_PARAMS, xaxis_title="Error (%)", yaxis_title="Probability Density"
)

mu, std = norm.fit(steady_state_df["Noise (%)"])
x_range = np.linspace(steady_state_df["Noise (%)"].min(), steady_state_df["Noise (%)"].max(), 100)
y_pdf = norm.pdf(x_range, mu, std)
fig.add_trace(
    go.Scatter(
        x=x_range, y=y_pdf, mode="lines", name="Gaussian Fit", line=dict(color="red")
    )
)

fig.show()

print(f"Mean: {mu:.5f}")
print(f"Std: {std:.5f}")

### Parameter Selection

- The raw sensor signal to noise ratio is very high
- It's possible that the sensor firmware is applying a low pass filter of its own
- For a first order IIR filter with $\alpha$ = 0.1:

$$
\begin{align}
\sigma_{out}^2 &= \sigma_{in}^2 \frac{\alpha}{2 - \alpha} \\\\
&\approx 0.007°
\end{align}
$$

and

$$
\begin{align}
\tau &= \frac{\Delta t}{a} \\\\
&= 20 s
\end{align}
$$

- The filter reduces the noise variance by ≈78% with a 20 s time constant

### Apply the Filter

In [ ]:
# Filter the signal and subtract the result from the measurements to approximately
# isolate the residual noise for statistical analysis
df["Filtered Humidity (EWM)"] = df["Humidity (%)"].ewm(alpha=0.015, adjust=False).mean()

display(df)

In [ ]:
fig = px.line(
    df, x=df.index, y=["Humidity (%)", "Filtered Humidity (EWM)"], title="Filtered Humidity"
)

for threshold in [20, 40, 50, 60, 70]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1
    )
fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_layout(xaxis_title="Elapsed Time", xaxis_tickformat="%H:%M:%S")

display(fig)

In [ ]:
time_window = ('00:32', '00:42')

fig = px.line(
    df.between_time(*time_window),
    x=df.between_time(*time_window).index,
    y=["Humidity (%)", "Filtered Humidity (EWM)"],
    title="Filtered Humidity (Zoomed)",
)

for threshold in [40]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_width=1
    )

fig.update_layout(LAYOUT_PARAMS)
fig.update_layout(xaxis_title="Elapsed Time", xaxis_tickformat="%H:%M:%S")

display(fig)

### Visualize the Patterns

In [ ]:
df["Humidity Pattern"] = df["Humidity (%)"].apply(render_pattern_sum_helper)
df["Filtered Humidity Pattern (EWM)"] = df["Filtered Humidity (EWM)"].apply(render_pattern_sum_helper)

In [ ]:
fig = px.line(
    df,
    x=df.index,
    y=[
        "Humidity Pattern",
        "Filtered Humidity Pattern (EWM)",
    ],
    title="LED Patterns",
    range_y=(0, 6)
)

fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Humidity Pattern (EWM)"},
)

fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_yaxes(title="Pattern Number")
fig.update_layout(xaxis_title="Elapsed Time", xaxis_tickformat="%H:%M:%S")

display(fig)

In [ ]:
fig = px.line(
    df.between_time(*time_window),
    x=df.between_time(*time_window).index,
    y=[
        "Humidity Pattern",
        "Filtered Humidity Pattern (EWM)",
    ],
    title="LED Patterns (Zoomed)",
    range_y=(1, 4)
)

fig.update_traces(
    line=dict(width=3),
    opacity=0.8,
    selector={"name": "Filtered Humidity Pattern (EWM)"},
)

fig.update_layout(dict1=LAYOUT_PARAMS)
fig.update_yaxes(title="Pattern Number", dtick=1)
fig.update_layout(xaxis_title="Elapsed Time", xaxis_tickformat="%H:%M:%S")

display(fig)

### Conclusion

- A filter parameter of $\alpha = 0.1$ successfully prevents LED pattern oscillations
- The resulting 20 s time constant does not introduce excessive lag